In [1]:
### Not completed
%cd /drive2/ryusejong/LFF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6,7"
import json 
import time 
import re
import random
import types
import math
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface, chat_huggingface_with_hidden_states, construct_conversation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from transformers.models.llama.modeling_llama import LlamaAttention
from gritlm import GritLM


seed = 42
set_seed(seed)

/drive2/ryusejong/LFF


/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load model and data

In [2]:
# Failure case & success case (each 20)
failure_path = "failure/FAILUREs20.jsonl"
success_path = "failure/SUCCESSes20.jsonl"

failure_data = read_data(failure_path)[0]
success_data = read_data(success_path)[0]

#print(f"Failure caes: {len(failure_data)}")
print(f"Failure caes: {failure_data}")
#print(f"Success caes: {len(success_data)}")
print(f"Success caes: {success_data}")

Failure caes: {'index': 19, 'question': "Janet filmed a new movie that is 60% longer than her previous 2-hour long movie.  Her previous movie cost $50 per minute to film, and the newest movie cost twice as much per minute to film as the previous movie.  What was the total amount of money required to film Janet's entire newest film?", 'answer': 1920.0, 'reasoning': 'The first movie was 2*60=<<2*60=120>>120 minutes\nSo this movie is 120*.6=<<120*.6=72>>72 minutes longer\nSo this movie is 192 minutes\nIt also cost 50*2=$<<50*2=100>>100 per minute to film\nSo it cost 192*100=$1920\n#### 1920', 'fail_answer': 25200.0, 'fail_reasoning': "Let's break this problem down step by step.\n\n1. The previous movie was 2 hours long, which is 120 minutes long.\n2. The new movie is 60% longer than the previous movie, so we need to find 60% of 120 minutes. To do this, we can convert 60% to a decimal by dividing by 100: 60/100 = 0.6. Then, we multiply 0.6 by 120 to get 0.6 x 120 = 72 minutes. This means t

In [3]:
# Load model (1)
HUGGINGFACE_TOKEN = "XXX"
model_path = "meta-llama/Meta-Llama-3-8B-Instruct"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda:0


In [4]:
# Load model (2)
tokenizer = AutoTokenizer.from_pretrained(model_path, token=HUGGINGFACE_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    token=HUGGINGFACE_TOKEN,
    torch_dtype=torch.float32,
    device_map="auto"
)
model.generation_config.temperature=None
model.generation_config.top_p=None
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:10<00:00,  2.66s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_

# Extract hidden states from failure and success (model.generate(), casual masking)

In [5]:
# Get failure output and hidden states
failure_QAs = dict()
failure_QAs['index'] = int(0)

failure_question = failure_data['question']
extractor = " Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."
failure_question = failure_question + " Explain your reasoning step-by-step." + extractor
    
failure_QAs['Q'] = {'role': 'user', 'content': failure_question}
failure_messages=[{'role': 'user', 'content': failure_question}]
    
failure_response, failure_response_tokens, failure_hidden_states = chat_huggingface_with_hidden_states(failure_messages, model, tokenizer, max_new_tokens=512)

User: Janet filmed a new movie that is 60% longer than her previous 2-hour long movie.  Her previous movie cost $50 per minute to film, and the newest movie cost twice as much per minute to film as the previous movie.  What was the total amount of money required to film Janet's entire newest film? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response.

Assistant: Let's break this problem down step by step.

1. The previous movie was 2 hours long, which is 120 minutes long.
2. The new movie is 60% longer than the previous movie, so we need to find 60% of 120 minutes. To do this, we can convert 60% to a decimal by dividing by 100: 60/100 = 0.6. Then, we multiply 0.6 by 120 to get 0.6 x 120 = 72 minutes. This means the new movie is 120 + 72 = 192 minutes long.
3. The previous movie cost $50 per minute to film, so the total cost of filming the previous movie was 120 minutes x $50 per minute

In [6]:
# Get success output and hidden states
success_QAs = dict()
success_QAs['index'] = int(0)

success_question = success_data['question']
extractor = " Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."
success_question = success_question + " Explain your reasoning step-by-step." + extractor
    
success_QAs['Q'] = {'role': 'user', 'content': success_question}
success_messages=[{'role': 'user', 'content': success_question}]
    
success_response, success_response_tokens, success_hidden_states = chat_huggingface_with_hidden_states(success_messages, model, tokenizer, max_new_tokens=512)

User: Silvia’s bakery is offering 10% on advanced orders over $50.00.  She orders 2 quiches for $15.00 each, 6 croissants at $3.00 each and 6 buttermilk biscuits for $2.00 each.  How much will her order be with the discount? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response.

Assistant: Let's break down Silvia's order step by step:

1. Quiches: 2 quiches at $15.00 each = 2 x $15.00 = $30.00
2. Croissants: 6 croissants at $3.00 each = 6 x $3.00 = $18.00
3. Buttermilk biscuits: 6 biscuits at $2.00 each = 6 x $2.00 = $12.00

Total cost before discount: $30.00 + $18.00 + $12.00 = $60.00

Since the order is over $50.00, Silvia is eligible for the 10% discount. To calculate the discount, multiply the total cost by the discount percentage:

$60.00 x 0.10 = $6.00

Discount amount: $6.00

Now, subtract the discount from the total cost to get the final amount:

$60.00 - $6.00 = $54.00

## 54 

In [7]:
# Check failure output and hidden states (1)
print(f"failure_response: {failure_response}")
print(f"failure_response_tokens: {len(failure_response_tokens)}")
print('\n')

for token in failure_response_tokens:
    print(tokenizer.decode(token, skip_special_tokens=True).strip(), end=' ')
print("\n")

print(f"seq_len: {len(failure_hidden_states)}")                                         # token seq_len
print(f"num_layer: {len(failure_hidden_states[0])}")                                  # num_leyars
print(f"failure_hidden_states: {failure_hidden_states[0][1].shape}")                    # (batch_size, seq_len or 1, hidden_size)

total_failure_hidden_states = []
for layer in range(1, len(failure_hidden_states[0])):                                   # num_layers
    layer_failure_hidden_states = []
    for seq in range(len(failure_hidden_states)):                                       # num_layers
        layer_failure_hidden_states.append(failure_hidden_states[seq][layer])           # (batch_size, prompt_len+1 or 1, hidden_size)
    layer_failure_hidden_states = torch.concatenate(layer_failure_hidden_states, dim=1) # (batch_size, seq_len, hidden_size)
    total_failure_hidden_states.append(layer_failure_hidden_states)
total_failure_hidden_states = torch.stack(total_failure_hidden_states, dim=0)           # (num_layers, batch_size, seq_len, hidden_size)

print(f"total_failure_hidden_states: {total_failure_hidden_states.shape}")              # (num_layers, batch_size, seq_len, hidden_size)

failure_response: Let's break this problem down step by step.

1. The previous movie was 2 hours long, which is 120 minutes long.
2. The new movie is 60% longer than the previous movie, so we need to find 60% of 120 minutes. To do this, we can convert 60% to a decimal by dividing by 100: 60/100 = 0.6. Then, we multiply 0.6 by 120 to get 0.6 x 120 = 72 minutes. This means the new movie is 120 + 72 = 192 minutes long.
3. The previous movie cost $50 per minute to film, so the total cost of filming the previous movie was 120 minutes x $50 per minute = $6000.
4. The new movie costs twice as much per minute to film as the previous movie, so it costs $100 per minute. The total cost of filming the new movie is 192 minutes x $100 per minute = $19,200.
5. To find the total amount of money required to film Janet's entire newest film, we add the cost of filming the new movie to the cost of filming the previous movie: $19,200 + $6000 = $25,200.

## 25,200 ##

Therefore, the total amount of money re

In [8]:
# Check success output and hidden states (1)
print(f"success_response: {success_response}")
print(f"success_response_tokens: {len(success_response_tokens)}")
print('\n')

for token in success_response_tokens:
    print(tokenizer.decode(token, skip_special_tokens=True).strip(), end=' ')
print("\n")

print(f"success_hidden_states: {len(success_hidden_states)}")                           # token seq_len
print(f"success_hidden_states: {len(success_hidden_states[0])}")                      # num_leyars
print(f"success_hidden_states: {success_hidden_states[1][1].shape}")                    # (batch_size, seq_len or 1, hidden_size)

total_success_hidden_states = []
for layer in range(1, len(success_hidden_states[0])):                                   # num_layers
    layer_success_hidden_states = []
    for seq in range(len(success_hidden_states)):                                       # num_layers
        layer_success_hidden_states.append(success_hidden_states[seq][layer])           # (batch_size, prompt_len+1 or 1, hidden_size)
    layer_success_hidden_states = torch.concatenate(layer_success_hidden_states, dim=1) # (batch_size, seq_len, hidden_size)
    total_success_hidden_states.append(layer_success_hidden_states)
total_success_hidden_states = torch.stack(total_success_hidden_states, dim=0)           # (num_layers, batch_size, seq_len, hidden_size)

print(f"total_success_hidden_states: {total_success_hidden_states.shape}")              # (num_layers, batch_size, seq_len, hidden_size)

success_response: Let's break down Silvia's order step by step:

1. Quiches: 2 quiches at $15.00 each = 2 x $15.00 = $30.00
2. Croissants: 6 croissants at $3.00 each = 6 x $3.00 = $18.00
3. Buttermilk biscuits: 6 biscuits at $2.00 each = 6 x $2.00 = $12.00

Total cost before discount: $30.00 + $18.00 + $12.00 = $60.00

Since the order is over $50.00, Silvia is eligible for the 10% discount. To calculate the discount, multiply the total cost by the discount percentage:

$60.00 x 0.10 = $6.00

Discount amount: $6.00

Now, subtract the discount from the total cost to get the final amount:

$60.00 - $6.00 = $54.00

## 54 ##

Final Answer: The final answer is 54. I hope it is correct.
success_response_tokens: 239


Let 's break down Sil via 's order step by step : 1 . Qu ich es :  2 qu ich es at $ 15 . 00 each =  2 x $ 15 . 00 = $ 30 . 00  2 . Cro iss ants :  6 cro iss ants at $ 3 . 00 each =  6 x $ 3 . 00 = $ 18 . 00  3 . But term ilk biscuits :  6 biscuits at $ 2 . 00 each =  6 x $ 2 . 00

# Extract hidden states from failure and success (model(), causal masking)

In [9]:
inputs = tokenizer(construct_conversation(failure_messages) + " " + failure_response, return_tensors="pt").to(device)
print(construct_conversation(failure_messages) + " " + failure_response)

with torch.no_grad():
    outputs = model(
        **inputs,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        output_hidden_states=True,
    )
entire_failure_hidden_states = torch.stack(outputs.hidden_states[1:], dim=0)
print(f"layer_num: {len(entire_failure_hidden_states)}")
print(f"failure hidden states: {entire_failure_hidden_states.shape}")

User: Janet filmed a new movie that is 60% longer than her previous 2-hour long movie.  Her previous movie cost $50 per minute to film, and the newest movie cost twice as much per minute to film as the previous movie.  What was the total amount of money required to film Janet's entire newest film? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response.

Assistant: Let's break this problem down step by step.

1. The previous movie was 2 hours long, which is 120 minutes long.
2. The new movie is 60% longer than the previous movie, so we need to find 60% of 120 minutes. To do this, we can convert 60% to a decimal by dividing by 100: 60/100 = 0.6. Then, we multiply 0.6 by 120 to get 0.6 x 120 = 72 minutes. This means the new movie is 120 + 72 = 192 minutes long.
3. The previous movie cost $50 per minute to film, so the total cost of filming the previous movie was 120 minutes x $50 per minute

In [10]:
inputs = tokenizer(construct_conversation(success_messages) + " " + success_response, return_tensors="pt").to(device)
print(construct_conversation(success_messages) + " " + success_response)
with torch.no_grad():
    outputs = model(
        **inputs,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        output_hidden_states=True
    )
entire_success_hidden_states = torch.stack(outputs.hidden_states[1:], dim=0)
print(f"layer_num: {len(entire_success_hidden_states)}")
print(f"success hidden states: {entire_success_hidden_states.shape}")

User: Silvia’s bakery is offering 10% on advanced orders over $50.00.  She orders 2 quiches for $15.00 each, 6 croissants at $3.00 each and 6 buttermilk biscuits for $2.00 each.  How much will her order be with the discount? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response.

Assistant: Let's break down Silvia's order step by step:

1. Quiches: 2 quiches at $15.00 each = 2 x $15.00 = $30.00
2. Croissants: 6 croissants at $3.00 each = 6 x $3.00 = $18.00
3. Buttermilk biscuits: 6 biscuits at $2.00 each = 6 x $2.00 = $12.00

Total cost before discount: $30.00 + $18.00 + $12.00 = $60.00

Since the order is over $50.00, Silvia is eligible for the 10% discount. To calculate the discount, multiply the total cost by the discount percentage:

$60.00 x 0.10 = $6.00

Discount amount: $6.00

Now, subtract the discount from the total cost to get the final amount:

$60.00 - $6.00 = $54.00

## 54 

In [11]:
print(total_failure_hidden_states[31, 0, 300, 200:230])
print(entire_failure_hidden_states[31, 0, 300, 200:230])

tensor([ 1.2627, -2.9762, -2.7970, -1.4685,  0.7203,  2.0967, -0.9456,  2.1567,
        -0.4093, -0.1456,  3.4173,  2.9632,  4.9837, -2.8367,  5.5221, -0.5079,
        -2.8137, -1.0938,  1.6209, -1.3903,  0.0888,  1.1299,  0.9281, -1.7570,
         2.1726,  2.6447,  2.4326,  1.4540,  3.4303, -1.7253], device='cuda:0')
tensor([ 1.2627, -2.9762, -2.7970, -1.4685,  0.7203,  2.0967, -0.9456,  2.1568,
        -0.4093, -0.1456,  3.4173,  2.9632,  4.9837, -2.8367,  5.5221, -0.5079,
        -2.8137, -1.0938,  1.6209, -1.3903,  0.0888,  1.1299,  0.9281, -1.7570,
         2.1725,  2.6447,  2.4326,  1.4540,  3.4303, -1.7253], device='cuda:0')


# Extract hidden states from failure and success (model(), no maksing)

In [12]:
# ─── 1. forward 함수 패치 ────────────────────────────
orig_forward = LlamaAttention.forward
def forward_no_causal(self, hidden_states, attention_mask=None,
                      position_ids=None, past_key_value=None,
                      output_attentions=False, use_cache=False,
                      **kwargs):
    # --- 원본 코드 일부 발췌 & 수정 ---
    bsz, q_len, _ = hidden_states.size()
    query_states = self.q_proj(hidden_states)
    key_states   = self.k_proj(hidden_states)
    value_states = self.v_proj(hidden_states)

    query_states = query_states.view(bsz, q_len, self.num_heads, -1)
    key_states   = key_states.view(bsz, q_len, self.num_heads, -1)
    value_states = value_states.view(bsz, q_len, self.num_heads, -1)

    # ► Flash-Attention / SDPA 호출 시 causal=False ◄
    attn_output = torch.nn.functional.scaled_dot_product_attention(
        query_states, key_states, value_states,
        attn_mask=None,
        is_causal=False                       # ← causal OFF
    )

    attn_output = attn_output.view(bsz, q_len, -1)
    attn_output = self.o_proj(attn_output)
    if not output_attentions:
        return attn_output, None, past_key_value
    else:
        return attn_output, None, past_key_value

# 덮어쓰기
#LlamaAttention.forward = forward_no_causal
for layer in model.model.layers:
    #layer.self_attn.forward = types.MethodType(forward_no_causal, layer.self_attn)
    layer.self_attn.forward = types.MethodType(LlamaAttention.forward, layer.self_attn)


In [13]:
inputs = tokenizer(construct_conversation(failure_messages) + " " + failure_response, return_tensors="pt").to(device)
inputs = {"input_ids": inputs.input_ids}    # eliminate attention mask
print(construct_conversation(failure_messages) + " " + failure_response)

with torch.no_grad():
    outputs = model(
        **inputs,
        use_cache=False,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        output_hidden_states=True,
    )
no_mask_entire_failure_hidden_states = torch.stack(outputs.hidden_states[1:], dim=0)
print(f"layer_num: {len(no_mask_entire_failure_hidden_states)}")
print(f"failure hidden states: {no_mask_entire_failure_hidden_states.shape}")

User: Janet filmed a new movie that is 60% longer than her previous 2-hour long movie.  Her previous movie cost $50 per minute to film, and the newest movie cost twice as much per minute to film as the previous movie.  What was the total amount of money required to film Janet's entire newest film? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response.

Assistant: Let's break this problem down step by step.

1. The previous movie was 2 hours long, which is 120 minutes long.
2. The new movie is 60% longer than the previous movie, so we need to find 60% of 120 minutes. To do this, we can convert 60% to a decimal by dividing by 100: 60/100 = 0.6. Then, we multiply 0.6 by 120 to get 0.6 x 120 = 72 minutes. This means the new movie is 120 + 72 = 192 minutes long.
3. The previous movie cost $50 per minute to film, so the total cost of filming the previous movie was 120 minutes x $50 per minute

In [14]:
print(entire_failure_hidden_states[31, 0, 300, 200:230])
print(no_mask_entire_failure_hidden_states[31, 0, 300, 200:230])

tensor([ 1.2627, -2.9762, -2.7970, -1.4685,  0.7203,  2.0967, -0.9456,  2.1568,
        -0.4093, -0.1456,  3.4173,  2.9632,  4.9837, -2.8367,  5.5221, -0.5079,
        -2.8137, -1.0938,  1.6209, -1.3903,  0.0888,  1.1299,  0.9281, -1.7570,
         2.1725,  2.6447,  2.4326,  1.4540,  3.4303, -1.7253], device='cuda:0')
tensor([ 1.2627, -2.9762, -2.7970, -1.4685,  0.7203,  2.0967, -0.9456,  2.1568,
        -0.4093, -0.1456,  3.4173,  2.9632,  4.9837, -2.8367,  5.5221, -0.5079,
        -2.8137, -1.0938,  1.6209, -1.3903,  0.0888,  1.1299,  0.9281, -1.7570,
         2.1725,  2.6447,  2.4326,  1.4540,  3.4303, -1.7253], device='cuda:0')
